# Debug: #385 — characters reordered around double line breaks

**Symptom** (from the issue): a lattice cell whose visual text is
```
ABC
DEF

GHI
```
comes out as `GABC\nDEF\nHI` — the first character after the blank line floats to the front.

**Root cause hypothesis:** `_process_horizontal_cut` iterates `textline._objs` in playa's emission order, which isn't reading order around blank lines.

**The speculative fix** (PR #758, closed because it broke `test_lattice_split_text`): sort LTChars by `(-y, x)` and synthesise line breaks on y-jumps. This notebook lets you decide whether the fix is correct (and the lattice fixture needs updating) or over-corrects.

Issue: [#385](https://github.com/camelot-dev/camelot/issues/385) · closed PR: #758

**Setup:** `pip install -e .[plot]`, then run.

## ⚙️ Colab bootstrap (run me first)

On Google Colab this clones the `debug/385-char-reading-order` branch, installs camelot editable, and cd-s in. On a local checkout it is a no-op.


In [ ]:
import sys, os, subprocess

BRANCH = "debug/385-char-reading-order"
REPO = "https://github.com/bosd/camelot.git"

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != "camelot" and not os.path.isdir("camelot/.git"):
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO], check=True)
    if os.path.basename(os.getcwd()) != "camelot":
        os.chdir("camelot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[plot]", "pytest"], check=True)
    print("Colab bootstrap complete -> cwd:", os.getcwd())
else:
    print("Not on Colab - assuming a local editable checkout; skipping bootstrap.")


In [ ]:
# Fetch the issue's reproducer PDF (1b.pdf).
import urllib.request, os
REPRO_URL = 'https://github.com/camelot-dev/camelot/files/11950066/1b.pdf'
os.makedirs('/tmp/camelot-385', exist_ok=True)
dst = '/tmp/camelot-385/1b.pdf'
if not os.path.exists(dst):
    try:
        urllib.request.urlretrieve(REPRO_URL, dst)
        print('downloaded', dst)
    except Exception as e:
        print('download failed:', e)
        print('Manually download from the issue and save to', dst)
else:
    print('already have', dst)

## Step 1 — reproduce the symptom on master

Extract with `flavor='lattice'` and look at the cell the issue flagged (row 6, 2nd column). You should see the `G`-floated-to-front scramble.

In [ ]:
import camelot
tables = camelot.read_pdf('/tmp/camelot-385/1b.pdf', flavor='lattice')
print('tables:', len(tables))
if tables:
    df = tables[0].df
    print(df.shape)
    # Print all cells so you can spot the scrambled one.
    for r in range(df.shape[0]):
        for c in range(df.shape[1]):
            val = df.iat[r, c]
            if '\n' in val:
                print(f'[{r},{c}] {val!r}')

## Step 2 — apply the speculative reading-order sort

Monkey-patch `_process_horizontal_cut` (or `_reading_order_chars` from the closed PR) and re-extract. Compare the flagged cell.

In [ ]:
# The closed PR #758's helper. Paste it here and wire it into _process_horizontal_cut.
from playa.miner import LTAnno, LTChar

def _reading_order_chars(objs):
    chars = [o for o in objs if isinstance(o, LTChar)]
    if not chars:
        yield from objs
        return
    chars.sort(key=lambda o: (-o.y0, o.x0))
    prev_y = None
    for ch in chars:
        if prev_y is not None and (prev_y - ch.y0) > (ch.y1 - ch.y0) * 0.5:
            yield LTAnno('\n')
        yield ch
        prev_y = ch.y0

import camelot.utils as U
_orig_hcut = U._process_horizontal_cut
import inspect
print(inspect.getsource(_orig_hcut))
# Replace the `for obj in textline._objs:` line with
# `for obj in _reading_order_chars(textline._objs):` in a patched copy.

## Step 3 — the deciding test

Run `test_lattice_split_text` WITH the patch applied:

```bash
python -m pytest tests/test_lattice.py::test_lattice_split_text -v
```

**Decision tree:**
- If 1b.pdf is now correct AND `test_lattice_split_text` still passes → the fix is good, open a clean PR.
- If 1b.pdf is correct but `test_lattice_split_text` fails → inspect the m27.pdf output by eye. If the *new* output is actually more correct than the fixture, the fix is right and the fixture needs updating (document why in the PR). If the new output is worse, the sort over-corrects → tighten the y-jump threshold or gate the behaviour.

In [ ]:
import subprocess
r = subprocess.run(['python','-m','pytest','tests/test_lattice.py','-v','--no-header'],
                   capture_output=True, text=True)
print(r.stdout[-3000:])